# 06 - Evaluation, confusion-matrix analysis and paper comparison

**What this notebook does**
1. Loads the three results files and shows them side by side.
2. Re-audits **every** run that has saved predictions against the current collapse detector - no
   retraining, no GPU, seconds (LESSON 5).
3. Builds the tumour-vs-healthy / subtype-vs-subtype breakdown for every model from those same saved
   predictions (LESSON 11) - the project's most defensible finding.
4. Builds the comparison figure and the paper-vs-replication table.
5. Writes `outputs/reports/comparison.md` with the computed numbers and clearly-marked placeholders
   for the narrative you write yourself.

**What must already exist**: notebooks 00 and 02-05. Anything missing is skipped with a warning
rather than failing.

**Cost**: no training at all. Everything here reads CSVs.

**What "looks right"**: `results_table.csv` with one row per model and no `INVALID_*` rows; a
confusion matrix per model with no all-zero columns; and a large positive gap between
tumour-vs-healthy and subtype accuracy for every model, MiniConvNet and baselines alike.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *

ensure_dirs()
print('outputs:', OUTPUT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluate_utils import (load_results, valid_only, format_mean_std, list_saved_predictions,
                                load_predictions, recheck_saved_predictions, compute_metrics,
                                confusion, plot_confusion_matrix, per_class_report,
                                tumor_vs_subtype_breakdown, interpret_breakdown, detect_collapse)

pd.set_option('display.width', 180)

## 1. The three results files

**Looks right**: `results_table.csv` = one row per model (canonical); `experiments_log.csv` = every
run with a `config_note`; `ablation_dropout.csv` = exactly 2 rows.

In [ ]:
canon = load_results('canonical')
exp = load_results('experiment')
abl = load_results('ablation')

print(f'results_table.csv    : {len(canon)} rows')
print(f'experiments_log.csv  : {len(exp)} rows')
print(f'ablation_dropout.csv : {len(abl)} rows')
print()
if len(canon):
    print(canon[['model', 'arch_variant', 'params', 'accuracy', 'accuracy_std', 'f1_macro',
                 'n_runs', 'status']].round(4).to_string(index=False))
else:
    print('results_table.csv is empty - run notebooks 03 and 05 first.')

In [ ]:
if len(canon):
    dupes = canon['model'][canon['model'].duplicated()].tolist()
    print('duplicate model rows:', dupes if dupes else 'none - good')
    bad = canon[canon['status'] != VALID_TAG]
    print('invalid rows in the canonical table:',
          bad['model'].tolist() if len(bad) else 'none - good')

## 2. Collapse audit over every saved run (LESSONS 4 + 5)

This applies the **current** detector to every run with raw predictions on disk, without retraining
anything. It is the check v2 could not perform on its own finished runs, and the reason
`save_predictions()` is called from the first run onward.

**Looks right**: every run `ok` with 4 classes predicted. Any `INVALID_partial_collapse` row names
exactly which classes were never predicted.

In [ ]:
saved = list_saved_predictions()
print(f'runs with raw predictions on disk: {len(saved)}')

if saved:
    recheck = recheck_saved_predictions(saved, verbose=False)
    print()
    print(recheck.to_string(index=False))
    invalid = recheck[recheck['status'] != VALID_TAG]
    print(f'\n{len(invalid)} of {len(recheck)} runs fail the current collapse check.')
    if len(invalid):
        print(invalid[['run_name', 'status', 'never_predicted_classes']].to_string(index=False))
else:
    recheck = pd.DataFrame()
    print('None yet - run notebooks 02-05.')

## 3. Model comparison (valid rows only)

Invalid rows are excluded before ranking - a broken model must never top the table.

In [ ]:
ranked = valid_only(canon).sort_values('accuracy', ascending=False) if len(canon) else canon
if len(ranked):
    print(ranked[['model', 'params', 'accuracy', 'accuracy_std', 'f1_macro', 'cohen_kappa',
                  'mcc']].round(4).to_string(index=False))
else:
    print('nothing to rank yet')

In [ ]:
if len(ranked):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    err = ranked['accuracy_std'].fillna(0).values
    axes[0].barh(ranked['model'], ranked['accuracy'], xerr=err, color='#4c78a8')
    axes[0].axvline(0.96, ls='--', color='crimson', lw=1.5, label="paper's 0.96")
    axes[0].axvline(0.25, ls=':', color='grey', lw=1, label='chance')
    axes[0].set_xlabel('test accuracy')
    axes[0].set_title('Accuracy by model (error bars = CV std where available)')
    axes[0].invert_yaxis()
    axes[0].legend()
    axes[0].grid(axis='x', alpha=0.3)

    axes[1].scatter(ranked['params'], ranked['accuracy'], s=60, color='#e45756')
    for _, r in ranked.iterrows():
        axes[1].annotate(r['model'], (r['params'], r['accuracy']), fontsize=7,
                         xytext=(4, 4), textcoords='offset points')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('parameters (log scale)')
    axes[1].set_ylabel('test accuracy')
    axes[1].set_title('Accuracy vs model size')
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    out = FIGURES_DIR / 'model_comparison.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('saved', out)

## 4. Tumour-vs-healthy versus subtype accuracy (LESSON 11)

The headline accuracy hides two very different tasks: detecting tumour at all, and telling the three
tumour subtypes apart. Across both previous attempts **every** model - including the baselines that
worked well - detected tumour at 92-99% while subtype discrimination sat at 35-66%. That gap, not a
training bug, is the story of this project.

Computed here straight from the saved predictions, so it costs nothing and covers every run.

**Looks right**: `detection_minus_subtype` large and positive for every model.

In [ ]:
rows = []
for run_name in saved:
    y_true, y_pred, y_prob = load_predictions(run_name)
    m = compute_metrics(y_true, y_pred, y_prob)
    b = tumor_vs_subtype_breakdown(y_true, y_pred)
    c = detect_collapse(kappa=m['cohen_kappa'], mcc=m['mcc'], y_pred=y_pred)
    rows.append({
        'run': run_name,
        'n': len(y_true),
        'overall_accuracy': round(m['accuracy'], 4),
        'tumor_vs_healthy': round(b['binary_tumor_vs_healthy_accuracy'], 4),
        'subtype_accuracy': round(b['subtype_accuracy_all_tumors'], 4),
        'detection_minus_subtype': round(b['binary_tumor_vs_healthy_accuracy']
                                         - b['subtype_accuracy_all_tumors'], 4),
        'most_over_predicted': b['most_over_predicted_class'],
        'never_predicted': ', '.join(c['details'].get('never_predicted_classes', [])) or '-',
        'status': c['status'],
    })
breakdown_tbl = pd.DataFrame(rows).sort_values('overall_accuracy', ascending=False)
print(breakdown_tbl.to_string(index=False) if len(breakdown_tbl) else 'no saved predictions yet')

In [ ]:
if len(breakdown_tbl):
    sub = breakdown_tbl.sort_values('overall_accuracy')
    fig, ax = plt.subplots(figsize=(10, 0.5 * len(sub) + 2.5))
    y = np.arange(len(sub))
    ax.barh(y - 0.2, sub['tumor_vs_healthy'], height=0.4, label='tumour vs healthy',
            color='#4c78a8')
    ax.barh(y + 0.2, sub['subtype_accuracy'], height=0.4, label='subtype (true tumours)',
            color='#e45756')
    ax.set_yticks(y)
    ax.set_yticklabels(sub['run'], fontsize=8)
    ax.set_xlabel('accuracy')
    ax.set_title('The two tasks, separated: detection is easy, subtyping is not (LESSON 11)')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    out = FIGURES_DIR / 'tumor_vs_subtype.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('saved', out)

## 5. Confusion matrices and per-class detail

Regenerated from saved predictions for the MiniConvNet runs - no checkpoint reloading, no forward
passes.

In [ ]:
FOCUS = [r for r in saved if r.startswith(('miniconvnet', 'cv_faithful_pooled'))]
print('detailed view for:', FOCUS if FOCUS else '(none found)')

analyses = {}
for run_name in FOCUS:
    y_true, y_pred, y_prob = load_predictions(run_name)
    m = compute_metrics(y_true, y_pred, y_prob)
    b = tumor_vs_subtype_breakdown(y_true, y_pred)
    c = detect_collapse(kappa=m['cohen_kappa'], mcc=m['mcc'], y_pred=y_pred)
    print('=' * 72)
    print(run_name)
    print('metrics:', {k: round(v, 4) for k, v in m.items()})
    print('status :', c['status'])
    print('\n' + interpret_breakdown(b))
    print('\nper-class report:')
    print(per_class_report(y_true, y_pred).round(4))
    plot_confusion_matrix(y_true, y_pred, run_name)
    analyses[run_name] = {'metrics': m, 'breakdown': b, 'collapse': c,
                          'cm': confusion(y_true, y_pred),
                          'interpretation': interpret_breakdown(b),
                          'never_predicted': c['details'].get('never_predicted_classes', [])}
    print()

## 6. Faithful vs clean: how much of the accuracy was leakage?

The `clean` split removes the duplicate-driven train/test leakage. The difference between the two is
a direct measure of what that leakage was worth - and it applies to the published number too, which
was computed on the same duplicated data.

In [ ]:
f = analyses.get('miniconvnet_faithful')
c = analyses.get('miniconvnet_clean')
if f and c:
    d = f['metrics']['accuracy'] - c['metrics']['accuracy']
    print(f"faithful={f['metrics']['accuracy']:.4f}  clean={c['metrics']['accuracy']:.4f}  "
          f"drop={d:+.4f}")
    print('\nReminder: the clean number is a robustness experiment, never presented as a '
          'replication of the paper.')
else:
    print('need both miniconvnet_faithful and miniconvnet_clean predictions - run notebook 02')

## 7. Paper vs replication

`PAPER_CLAIMS` holds only what the **paper itself reports** - nothing is invented and nothing is
pre-filled on the replication side. Entries left as `None` print as blank.

In [ ]:
PAPER_CLAIMS = {
    'MiniConvNet': {'accuracy': 0.96, 'params': 500_000},
    'ResNet50': {'accuracy': None, 'params': None},
    'VGG16': {'accuracy': None, 'params': None},
    'MobileNetV3Small': {'accuracy': None, 'params': None},
    'EfficientNetV2B0': {'accuracy': None, 'params': None},
}

OUR_ROW_FOR = {
    'MiniConvNet': 'MiniConvNet (3-fold CV)',
    'ResNet50': 'ResNet50',
    'VGG16': 'VGG16',
    'MobileNetV3Small': 'MobileNetV3Small',
    'EfficientNetV2B0': 'EfficientNetV2B0',
}

def our_number(model_name):
    if not len(canon):
        return None
    hit = canon[canon['model'] == model_name]
    if not len(hit):
        return None
    r = hit.iloc[0]
    if r['status'] != VALID_TAG:
        return str(r['status'])
    if pd.notna(r.get('accuracy_std')) and r.get('n_runs', 1) and r['n_runs'] > 1:
        return format_mean_std(r['accuracy'], r['accuracy_std'])
    return f"{r['accuracy']:.4f}"

comparison = pd.DataFrame([
    {'model': k,
     'paper_accuracy': v['accuracy'],
     'our_accuracy': our_number(OUR_ROW_FOR[k]),
     'paper_params': v['params'],
     'our_params': (canon.loc[canon['model'] == OUR_ROW_FOR[k], 'params'].iloc[0]
                    if len(canon) and (canon['model'] == OUR_ROW_FOR[k]).any() else None)}
    for k, v in PAPER_CLAIMS.items()])
print(comparison.to_string(index=False))
print(f'\nOur MiniConvNet figure is the {CV_FOLDS}-fold CV mean +/- std on the faithful split '
      '(3 folds, not 5: a disclosed CPU-budget decision - state this whenever you quote it).')

## 8. Write `outputs/reports/comparison.md`

The report carries the computed numbers plus `TODO:` markers where a human writes the
interpretation. Nothing narrative is auto-invented.

In [ ]:
lines = [
    '# Paper vs replication (v3)',
    '',
    'Paper: Baqir, M.A., Qayyum, S., Ashfaq, N. et al. "A lightweight CNN for enhanced non-small '
    'cell lung cancer classification using CT scan image." Scientific Reports 16, 12985 (2026). '
    'DOI: 10.1038/s41598-026-41401-w',
    '',
    'Scope: CPU-only. One MiniConvNet architecture (~499K params, the Flatten reading); '
    f'{CV_FOLDS}-fold cross-validation; four frozen-backbone baselines. Each of these is a '
    'disclosed budget decision, not an oversight.',
    '',
    '## Headline comparison',
    '',
    '```', comparison.to_string(index=False), '```',
    '',
    '## Canonical results table',
    '',
    '```',
    (canon.to_string(index=False) if len(canon) else '(empty - run notebooks 03 and 05)'),
    '```',
    '',
    '## Dropout ablation',
    '',
    '```',
    (abl.to_string(index=False) if len(abl) else '(empty - run notebook 04)'),
    '```',
    '',
    '## Collapse audit (LESSON 4)',
    '',
    'Two distinct failure tags. `INVALID_collapsed`: flat training curve, kappa/MCC ~0, or a single '
    'predicted class. `INVALID_partial_collapse`: the model never predicts some of the four classes '
    '- whole all-zero columns in the confusion matrix - which can happen at a clearly nonzero kappa '
    'and was the unresolved bug of the v2 attempt. Neither is a reportable result.',
    '',
    '```',
    (recheck.to_string(index=False) if len(recheck) else '(no saved predictions yet)'),
    '```',
    '',
    '## The two tasks, separated (LESSON 11)',
    '',
    'Tumour-vs-healthy accuracy against subtype accuracy, for every run:',
    '',
    '```',
    (breakdown_tbl.to_string(index=False) if len(breakdown_tbl) else '(no saved predictions yet)'),
    '```',
    '',
    '## Confusion matrices',
    '',
]
for name, a in analyses.items():
    lines += [
        f'### {name}', '',
        f"- overall accuracy: {a['metrics']['accuracy']:.4f}",
        f"- tumour-vs-healthy accuracy: {a['breakdown']['binary_tumor_vs_healthy_accuracy']:.4f}",
        f"- subtype accuracy (true tumour samples): {a['breakdown']['subtype_accuracy_all_tumors']:.4f}",
        f"- most over-predicted class: {a['breakdown']['most_over_predicted_class']}",
        f"- classes never predicted: {', '.join(a['never_predicted']) if a['never_predicted'] else 'none'}",
        f"- status: {a['collapse']['status']}", '',
    ]
    if a['collapse']['status'] != VALID_TAG:
        lines += ['**This run is invalid** - the numbers above are an audit trail, not a result.', '']
    lines += [a['interpretation'], '', '```', str(a['cm']), '```', '']

lines += [
    '## Interpretation (write these yourself)',
    '',
    "- TODO: how far is the replication from the paper's 96%, and is the remaining gap a "
    'tumour-detection problem or a subtype-discrimination problem (see the table above)?',
    '- TODO: how much of the faithful-split accuracy was duplicate-driven leakage (faithful vs '
    'clean)?',
    '- TODO: did the five lesson-2 measures eliminate the v2 partial collapse? What did the '
    'dead-unit probe in notebook 02 say?',
    '- TODO: how does a ~499K-parameter model trained from scratch compare with frozen ImageNet '
    'backbones 2-50x its size, on identical data?',
    '- TODO: if the result plateaued below the paper, say so plainly with this evidence. Do not '
    'describe the paper as replicated.',
]

report_path = REPORTS_DIR / 'comparison.md'
report_path.write_text('\n'.join(lines))
print('wrote', report_path)
print()
print('\n'.join(lines[:16]))

In [ ]:
print('artefacts:')
print('  figures    :', FIGURES_DIR)
print('  history    :', HISTORY_DIR)
print('  predictions:', PREDICTIONS_DIR)
print('  reports    :', REPORTS_DIR)
print('  tables     :', RESULTS_TABLE_CSV.name, ',', EXPERIMENTS_LOG_CSV.name, ',', ABLATION_CSV.name)
print('\nFill in the README Results and Paper-vs-replication sections from these files - '
      'do not hand-write numbers the pipeline did not produce.')